In [1]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [2]:
seed = 42
shuffle = True
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 10
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [3]:
load_model = efficientnet_b0(
     weights=EfficientNet_B0_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes)
model = model.to(device)

In [4]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [5]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]
#
df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

#### view data

In [6]:
(df_train.shape, df_val.shape, df_test.shape)

((7219, 5), (1805, 5), (1592, 4))

In [7]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    Albu.Blur(p=0.5),
])

In [8]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

In [9]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [10]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [11]:
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b0-entropy-trans.txt",
    path_to_save_model="models/b0-entropy-trans.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 301/301 [01:34<00:00,  3.19it/s]


VAL_LOSS     0.358
VAL_ACC      Mean: 39.396 | Std: 1.129 | 95% CI: [37.618, 41.330]
VAL_KAPPA    Mean: 0.705 | Std: 0.013 | 95% CI: [0.683, 0.726]
VAL_F1       Mean: 0.332 | Std: 0.011 | 95% CI: [0.314, 0.350]
VAL_RECALL   Mean: 0.344 | Std: 0.011 | 95% CI: [0.327, 0.361]
VAL_PRECISION Mean: 0.443 | Std: 0.019 | 95% CI: [0.411, 0.473]
Salvando o melhor modelo... 0.0 -> 0.7051458656416366
Epoch 2/50



100%|██████████| 301/301 [01:32<00:00,  3.25it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.303
VAL_ACC      Mean: 49.716 | Std: 1.173 | 95% CI: [47.867, 51.801]
VAL_KAPPA    Mean: 0.757 | Std: 0.012 | 95% CI: [0.737, 0.775]
VAL_F1       Mean: 0.407 | Std: 0.011 | 95% CI: [0.389, 0.425]
VAL_RECALL   Mean: 0.424 | Std: 0.011 | 95% CI: [0.407, 0.442]
VAL_PRECISION Mean: 0.549 | Std: 0.010 | 95% CI: [0.534, 0.565]
Salvando o melhor modelo... 0.7051458656416366 -> 0.7567133587304341
Epoch 3/50



...

VAL_LOSS     0.399
VAL_ACC      Mean: 52.251 | Std: 1.178 | 95% CI: [50.468, 54.183]
VAL_KAPPA    Mean: 0.764 | Std: 0.013 | 95% CI: [0.742, 0.786]
VAL_F1       Mean: 0.430 | Std: 0.012 | 95% CI: [0.411, 0.450]
VAL_RECALL   Mean: 0.443 | Std: 0.011 | 95% CI: [0.426, 0.461]
VAL_PRECISION Mean: 0.529 | Std: 0.017 | 95% CI: [0.500, 0.557]
Epoch 6/50



100%|██████████| 301/301 [01:42<00:00,  2.92it/s]


VAL_LOSS     0.408
VAL_ACC      Mean: 55.631 | Std: 1.204 | 95% CI: [53.629, 57.673]
VAL_KAPPA    Mean: 0.782 | Std: 0.013 | 95% CI: [0.760, 0.803]
VAL_F1       Mean: 0.479 | Std: 0.012 | 95% CI: [0.459, 0.500]
VAL_RECALL   Mean: 0.490 | Std: 0.011 | 95% CI: [0.473, 0.509]
VAL_PRECISION Mean: 0.557 | Std: 0.014 | 95% CI: [0.532, 0.579]
Salvando o melhor modelo... 0.7685074857328993 -> 0.7817216360124368
Epoch 7/50



loss: 0.09248, smooth loss: 0.11997:  10%|█         | 124/1204 [01:12<10:34,  1.70it/s]


KeyboardInterrupt: 

# tests

In [11]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b0-entropy-trans.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

100%|██████████| 199/199 [01:15<00:00,  2.64it/s]


VAL_ACC      Mean: 62.51 | Std: 1.22 | 95% CI: [60.49, 64.45]
VAL_KAPPA    Mean: 0.83 | Std: 0.01 | 95% CI: [0.80, 0.85]
VAL_F1       Mean: 0.57 | Std: 0.01 | 95% CI: [0.55, 0.59]
VAL_RECALL   Mean: 0.57 | Std: 0.01 | 95% CI: [0.55, 0.59]
VAL_PRECISION Mean: 0.57 | Std: 0.01 | 95% CI: [0.55, 0.59]
